In [14]:
from google.colab import drive
drive.mount('/content/drive')

import os

CHECKPOINT_DIR = "/content/drive/MyDrive/VoxIntegrity_Data/checkpoints"

print([
    f for f in os.listdir(CHECKPOINT_DIR)
    if "epoch1" in f
])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['ast_epoch1_batch500.pt', 'ast_epoch1_batch1000.pt', 'ast_epoch1_batch1500.pt', 'ast_epoch1_batch2000.pt', 'ast_epoch1_batch2500.pt', 'ast_epoch1_batch3000.pt', 'ast_epoch1_batch3500.pt', 'ast_epoch1_batch4000.pt', 'ast_epoch1_batch4500.pt', 'ast_epoch1_batch5000.pt', 'ast_epoch1_batch5500.pt', 'ast_epoch1_batch6000.pt']


In [15]:
!pip -q install transformers librosa soundfile scikit-learn

import os
import torch
import librosa
import numpy as np

from torch.utils.data import Dataset, DataLoader
from transformers import ASTForAudioClassification

print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

CUDA: True
GPU: Tesla T4


In [16]:
from google.colab import drive
drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/VoxIntegrity_Data/ASVspoof2019_LA"

DEV_DIR = os.path.join(BASE, "ASVspoof2019_LA_dev", "flac")

DEV_PROTOCOL = os.path.join(
    BASE,
    "ASVspoof2019_LA_cm_protocols",
    "ASVspoof2019.LA.cm.dev.trl.txt"
)

CHECKPOINT_PATH = os.path.join(
    "/content/drive/MyDrive/VoxIntegrity_Data/checkpoints",
    "ast_epoch1_batch6000.pt"
)

print("DEV exists:", os.path.exists(DEV_DIR))
print("Protocol exists:", os.path.exists(DEV_PROTOCOL))
print("Checkpoint exists:", os.path.exists(CHECKPOINT_PATH))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DEV exists: True
Protocol exists: True
Checkpoint exists: True


In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Using device: cuda
GPU: Tesla T4


In [18]:
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification

MODEL_DIR = "/content/drive/MyDrive/VoxIntegrity_Data/models/AST-ASVspoof2019"

feature_extractor = AutoFeatureExtractor.from_pretrained(
    MODEL_DIR,
    local_files_only=True
)

model = AutoModelForAudioClassification.from_pretrained(
    MODEL_DIR,
    local_files_only=True
).to(device)

print("Original AST loaded")
print("Labels:", model.config.id2label)

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

Original AST loaded
Labels: {0: 'Bonafide', 1: 'Spoof'}


In [19]:
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Fine-tune A checkpoint loaded successfully")
print("Epoch:", checkpoint["epoch"])
print("Batch:", checkpoint["batch"])

Fine-tune A checkpoint loaded successfully
Epoch: 1
Batch: 6000


In [20]:
import pandas as pd

dev_rows = []

with open(DEV_PROTOCOL, "r") as f:
    for line in f:
        parts = line.strip().split()

        if len(parts) >= 5:
            dev_rows.append({
                "audio_id": parts[1],
                "audio_path": os.path.join(DEV_DIR, parts[1] + ".flac"),
                "label": parts[4]
            })

dev_df = pd.DataFrame(dev_rows)

print("DEV samples:", len(dev_df))
print("\nLabel distribution:")
print(dev_df["label"].value_counts())

print("\nMissing audio files:",
      (~dev_df["audio_path"].apply(os.path.exists)).sum())

DEV samples: 24844

Label distribution:
label
spoof       22296
bonafide     2548
Name: count, dtype: int64

Missing audio files: 0


In [21]:
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

class ASVSpoofDataset(Dataset):
    def __init__(self, dataframe):
        self.data = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        waveform, sr = librosa.load(
            row["audio_path"],
            sr=16000,
            mono=True
        )

        max_length = 16000 * 4

        if len(waveform) > max_length:
            waveform = waveform[:max_length]
        else:
            waveform = F.pad(
                torch.tensor(waveform),
                (0, max_length - len(waveform))
            ).numpy()

        inputs = feature_extractor(
            waveform,
            sampling_rate=16000,
            return_tensors="pt"
        )

        label = 0 if row["label"] == "bonafide" else 1

        return {
            "input_values": inputs["input_values"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long)
        }


dev_dataset = ASVSpoofDataset(dev_df)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0
)

print("DEV Dataset:", len(dev_dataset))
print("DEV Batches:", len(dev_loader))

DEV Dataset: 24844
DEV Batches: 6211


In [22]:
# no need to run this

# from sklearn.metrics import accuracy_score, confusion_matrix
# from scipy.special import softmax

# EVAL_SAVE = "/content/drive/MyDrive/VoxIntegrity_Data/dev_eval_A_checkpoint.npz"

# all_labels = []
# all_predictions = []
# all_scores = []

# model.eval()

# with torch.no_grad():

#     for batch_idx, batch in enumerate(dev_loader):

#         input_values = batch["input_values"].to(device)
#         labels = batch["labels"].to(device)

#         outputs = model(input_values=input_values)

#         logits = outputs.logits
#         predictions = torch.argmax(logits, dim=1)
#         scores = softmax(
#             logits.cpu().numpy(),
#             axis=1
#         )[:, 1]

#         all_labels.extend(labels.cpu().numpy())
#         all_predictions.extend(predictions.cpu().numpy())
#         all_scores.extend(scores)

#         # Save checkpoint every 100 batches
#         if (batch_idx + 1) % 100 == 0:

#             np.savez(
#                 EVAL_SAVE,
#                 labels=np.array(all_labels),
#                 predictions=np.array(all_predictions),
#                 scores=np.array(all_scores),
#                 last_batch=batch_idx + 1
#             )

#             # Calculate LIVE metrics after saving
#             current_accuracy = accuracy_score(
#                 all_labels,
#                 all_predictions
#             )

#             cm = confusion_matrix(
#                 all_labels,
#                 all_predictions
#             )

#             print("\n" + "=" * 60)
#             print(f"CHECKPOINT SAVED: Batch {batch_idx + 1}/{len(dev_loader)}")
#             print(f"Samples evaluated: {len(all_labels)}")
#             print(f"Live Accuracy: {current_accuracy * 100:.2f}%")
#             print("Confusion Matrix:")
#             print(cm)
#             print("=" * 60)

# print("\nDEV evaluation completed.")
# print("Total samples:", len(all_labels))

KeyboardInterrupt: 

In [23]:
!pip -q install onnxscript

In [24]:
# Install onnx first
!pip install onnx

import os
import torch
import onnx
from transformers import AutoModelForAudioClassification

# --------------------------------------------------
# PATH CONFIGURATION
# --------------------------------------------------
MODEL_DIR = "/content/drive/MyDrive/VoxIntegrity_Data/models/AST-ASVspoof2019"
CHECKPOINT_PATH = "/content/drive/MyDrive/VoxIntegrity_Data/checkpoints/ast_epoch1_batch6000.pt"
OUTPUT_DIR = "/content/drive/MyDrive/VoxIntegrity_Data/exported_models"
os.makedirs(OUTPUT_DIR, exist_ok=True)

PTH_PATH = os.path.join(OUTPUT_DIR, "ast_finetuned_fp32.pth")
ONNX_PATH = os.path.join(OUTPUT_DIR, "ast_finetuned_fp32.onnx")

device = torch.device("cpu")  # Export CPU pe safe & clean graph produce karta hai

# --------------------------------------------------
# 1. LOAD MODEL & WEIGHTS (FP32)
# --------------------------------------------------
print("Loading model and checkpoint...")
model = AutoModelForAudioClassification.from_pretrained(
    MODEL_DIR,
    local_files_only=True
)

checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(device).float()  # Ensure strict FP32 precision
model.eval()

# --------------------------------------------------
# 2. EXPORT CLEAN PYTORCH (.pth)
# --------------------------------------------------
torch.save({
    "model_state_dict": model.state_dict(),
    "config": model.config,
    "id2label": model.config.id2label,
    "label2id": model.config.label2id
}, PTH_PATH)
print(f"PyTorch FP32 model saved to: {PTH_PATH}")
print(f"Size: {os.path.getsize(PTH_PATH) / (1024 * 1024):.2f} MB")

# --------------------------------------------------
# 3. EXPORT TO ONNX (FP32, DYNAMIC BATCH)
# --------------------------------------------------
# Shape matches dataset feature extraction: [Batch, Time_Frames=1024, Mel_Bins=128]
dummy_input = torch.randn(1, 1024, 128, dtype=torch.float32, device=device)

# Wrapper class so output is clean tensor logits instead of HF ModelOutput tuple
class ASTInferenceWrapper(torch.nn.Module):
    def __init__(self, hf_model):
        super().__init__()
        self.hf_model = hf_model

    def forward(self, input_values):
        outputs = self.hf_model(input_values=input_values)
        return outputs.logits

wrapper = ASTInferenceWrapper(model)
wrapper.eval()

print("\nExporting ONNX graph...")
torch.onnx.export(
    wrapper,
    dummy_input,
    ONNX_PATH,
    export_params=True,
    opset_version=14,                   # Stable for Transformer attention blocks
    do_constant_folding=True,
    input_names=["input_values"],
    output_names=["logits"],
    dynamic_axes={
        "input_values": {0: "batch_size"},
        "logits": {0: "batch_size"}
    }
)

# --------------------------------------------------
# 4. VERIFY ONNX INTEGRITY
# --------------------------------------------------
onnx_model = onnx.load(ONNX_PATH)
onnx.checker.check_model(onnx_model)
print(f"ONNX FP32 model saved & verified: {ONNX_PATH}")
print(f"Size: {os.path.getsize(ONNX_PATH) / (1024 * 1024):.2f} MB")

Loading model and checkpoint...


Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

/tmp/ipykernel_1226/2779061459.py:68: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0904 12:08:15.408000 1226 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 14 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


PyTorch FP32 model saved to: /content/drive/MyDrive/VoxIntegrity_Data/exported_models/ast_finetuned_fp32.pth
Size: 328.87 MB

Exporting ONNX graph...
[torch.onnx] Obtain model graph for `ASTInferenceWrapper([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ASTInferenceWrapper([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...


[torch.onnx] Translate the graph into ONNX... ✅


Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        proto, target_version=self.target_version
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
RuntimeError: /project/onnx/version_converter/adapters/no_previous_version.h:24: adap

[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
ONNX FP32 model saved & verified: /content/drive/MyDrive/VoxIntegrity_Data/exported_models/ast_finetuned_fp32.onnx
Size: 1.70 MB


In [25]:
import os
import onnx

ONNX_PATH = "/content/drive/MyDrive/VoxIntegrity_Data/exported_models/ast_finetuned_fp32.onnx"

onnx_model = onnx.load(ONNX_PATH)

print("ONNX file size:",
      f"{os.path.getsize(ONNX_PATH) / (1024 * 1024):.2f} MB")

print("IR version:", onnx_model.ir_version)
print("Opset:", onnx_model.opset_import[0].version)
print("Inputs:", [x.name for x in onnx_model.graph.input])
print("Outputs:", [x.name for x in onnx_model.graph.output])
print("Initializers:", len(onnx_model.graph.initializer))
print("Graph nodes:", len(onnx_model.graph.node))

total_params = sum(
    np.prod(init.dims)
    for init in onnx_model.graph.initializer
)

print("ONNX parameters:", total_params)

ONNX file size: 1.70 MB
IR version: 10
Opset: 18
Inputs: ['input_values']
Outputs: ['logits']
Initializers: 222
Graph nodes: 576
ONNX parameters: 86191570.0


In [26]:
!pip -q install onnxruntime

import os
import numpy as np
import torch
import onnxruntime as ort

ONNX_PATH = "/content/drive/MyDrive/VoxIntegrity_Data/exported_models/ast_finetuned_fp32.onnx"

# ------------------------------------------------------------
# 1. Load ONNX
# ------------------------------------------------------------
session = ort.InferenceSession(
    ONNX_PATH,
    providers=["CPUExecutionProvider"]
)

print("ONNX providers:", session.get_providers())

# ------------------------------------------------------------
# 2. Create identical test input
# ------------------------------------------------------------
np.random.seed(42)

test_input = np.random.randn(
    1, 1024, 128
).astype(np.float32)

# ------------------------------------------------------------
# 3. PyTorch inference
# ------------------------------------------------------------
model.eval()
model.cpu()

with torch.no_grad():
    pytorch_logits = model(
        torch.from_numpy(test_input)
    ).logits.numpy()

# ------------------------------------------------------------
# 4. ONNX inference
# ------------------------------------------------------------
onnx_logits = session.run(
    ["logits"],
    {"input_values": test_input}
)[0]

# ------------------------------------------------------------
# 5. Compare
# ------------------------------------------------------------
print("\nPyTorch logits:")
print(pytorch_logits)

print("\nONNX logits:")
print(onnx_logits)

print("\nShapes:")
print("PyTorch:", pytorch_logits.shape)
print("ONNX   :", onnx_logits.shape)

max_diff = np.max(
    np.abs(pytorch_logits - onnx_logits)
)

mean_diff = np.mean(
    np.abs(pytorch_logits - onnx_logits)
)

print("\nMaximum absolute difference:", max_diff)
print("Mean absolute difference:", mean_diff)

print(
    "\nOutputs close:",
    np.allclose(
        pytorch_logits,
        onnx_logits,
        rtol=1e-3,
        atol=1e-4
    )
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 22.3 MB/s eta 0:00:00
ONNX providers: ['CPUExecutionProvider']

PyTorch logits:
[[-0.9818624  2.5255148]]

ONNX logits:
[[-0.9818617  2.5255144]]

Shapes:
PyTorch: (1, 2)
ONNX   : (1, 2)

Maximum absolute difference: 7.1525574e-07
Mean absolute difference: 5.9604645e-07

Outputs close: True
